# pull FAERS quarterly data
* e.g. https://fis.fda.gov/content/Exports/faers_ascii_2023q2.zip

In [5]:
# pip install matplotlib

In [1]:
import os, re, time, shutil, pickle, json
import warnings, pickle, gc
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from io import BytesIO
from zipfile import ZipFile
from datetime import datetime
from bs4 import BeautifulSoup
from urllib.request import urlopen

In [16]:
# explore FAERS ASCII files
rpse_ = []
with open("/home/dada/Barn/GQ/ADR/hybrid_rag/faers/2025q2/ASCII/RPSR25Q2.txt", "r") as f:
    for line in f:
        rpse = line.strip()
        rpse_.append(rpse.split("$"))

In [18]:
len(rpse_)

11106

In [19]:
demo_df = pd.read_csv("/home/dada/Barn/GQ/ADR/hybrid_rag/faers/2025q2/ASCII/DEMO25Q2.txt", sep = "$", low_memory=False)

In [ ]:
len(rpse_)/demo_df.shape[0] #only ~3% ADR records have source indicator, USELESS! 

0.028250197135807495

In [4]:
#pip install tensorrt

In [2]:
pd.set_option("display.max_columns",  None)

## prepare quarterly string for FAERS since q4 2012

In [3]:
quarter_string = []
for i in range(2004, 2026):
    for j in ["q1", "q2", "q3", "q4"]:
        quarter_string.append(str(i)+j)

In [4]:
#get FAERS 2012q4 index
faers_start = quarter_string.index("2012q4")
faers_start

35

In [126]:
quarter_string[-5:-2] #oot for hybrid rag ACL 

['2024q4', '2025q1', '2025q2']

## download quarterly Zip files and unzip to ./faers/folder 

In [48]:
#wget -q https://fis.fda.gov/content/Exports/faers_ascii_{i}.zip -P ./faers/{i}

In [7]:
#for i in tqdm(quarter_string[faers_start:]):
for i in tqdm(quarter_string[-5:-3]):    
    try:
        str_in = f"https://fis.fda.gov/content/Exports/faers_ascii_{i}.zip"
        r = requests.get(str_in, timeout=200)

        #unzip
        z = ZipFile(BytesIO(r.content))
        z.extractall(f"./faers/{i}")
    except:
        continue

100%|█████████████████████████████████████████████| 2/2 [02:22<00:00, 71.11s/it]


## download quarterly data from AERS between Q1 2004 and q4 2012
* https://fis.fda.gov/content/Exports/aers_ascii_2012q3.zip
* FAERS dictionary @  https://pharmahub.org/app/site/resources/2018/01/00739/FDA-FAERS-Data-Dictionary.pdf

In [15]:
for i in tqdm(quarter_string[:faers_start]):
    str_in = f"https://fis.fda.gov/content/Exports/aers_ascii_{i}.zip"
    r = requests.get(str_in, timeout=200)
    z = ZipFile(BytesIO(r.content))
    z.extractall(f"./aers/{i}")

# prepare merge path_in string

In [5]:
path_in = []
for i in quarter_string[faers_start:-2]:   
#for i in quarter_string[faers_start:]:   
    if "ascii" in os.listdir(f"../faers/{i}"):
        path_in.append(f"../faers/{i}/ascii")
    elif "ASCII" in os.listdir(f"../faers/{i}"):
        path_in.append(f"../faers/{i}/ASCII")

In [128]:
path_in[-3:]

['../faers/2024q4/ASCII', '../faers/2025q1/ASCII', '../faers/2025q2/ASCII']

# get aers path

In [129]:
aers_in = []
for i in quarter_string[:faers_start]:   
    if "ascii" in os.listdir(f"../aers_merge/aers/{i}"):
        aers_in.append(f"../aers_merge/aers/{i}/ascii")
    elif "ASCII" in os.listdir(f"../aers_merge/aers/{i}"):
        aers_in.append(f"../aers_merge/aers/{i}/ASCII")

In [130]:
aers_in[-4:]

['../aers_merge/aers/2011q4/ascii',
 '../aers_merge/aers/2012q1/ascii',
 '../aers_merge/aers/2012q2/ascii',
 '../aers_merge/aers/2012q3/ascii']

In [6]:
def shiftCol(df):
    df_col = df.columns
    df.reset_index(inplace = True)
    df = df.iloc[:, :len(df_col)]
    #df.columns = [i.lower() for i in df_col]    
    df.columns = df_col    
    return df

# get report source (< 0.5%)

In [14]:
txt_src = "/home/dada/Barn/GQ/ADR/hybrid_rag/faers/2024q4/ASCII/RPSR24Q4.txt"

In [16]:
rpt_src = []
if "RPSR" in txt_src.upper() and "TXT" in txt_src.upper():
    try:
        rpt_src_df = pd.read_csv(txt_src, sep = "$", low_memory=False)
    except:
        rpt_src_df = pd.read_csv(txt_src, sep = "$", encoding='iso-8859-1', low_memory=False)   
        
    rpt_src += list(set(rpt_src_df.rpsr_cod.dropna()))

In [17]:
rpt_src

['FGN', 'CSM', 'HP']

In [ ]:
rpt_src_df.shape #out of 2030938, too few records have source indicator, USELESS! 

(11627, 3)

# create data frame w/o merged caseid

In [1]:
import os, json

In [2]:
# for filename in os.listdir(path_in):
#     if "DRUG" in filename.upper() and "TXT" in filename.upper():
#         try:
#             df = pd.read_csv(path_in[1] + "/" + filename, sep = "$", low_memory=False)
#         except:
#             df = pd.read_csv(path_in[1] + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)       

In [5]:
df = pd.read_csv("./faers/2023q4/ASCII/DRUG23Q4.txt", sep = "$", low_memory=False)

In [6]:
df = df[['primaryid','caseid', 'drugname','route','dose_vbm','dose_amt',
                               'dose_unit','dose_form','dose_freq']]

df = df[(df.drugname.isnull() == False) & (df.drugname != "nan")]           

#df['treatment'] = [i.to_dict() for _, i in drug_df.iterrows()]

In [7]:
df.head()

,primaryid,caseid,drugname,route,dose_vbm,dose_amt,dose_unit,dose_form,dose_freq
0,100144838,10014483,ISENTRESS,Transplacental,"400 mg, bid",400.0,MG,Tablet,BID
1,100144838,10014483,ISENTRESS,Transplacental,400 mg,400.0,MG,Tablet,NaN
2,100144838,10014483,ISENTRESS,Transplacental,UNK,NaN,NaN,Tablet,NaN
3,100144838,10014483,ABACAVIR,Transplacental,"300 mg, bid",300.0,MG,Capsule,BID
4,100144838,10014483,ABACAVIR,Transplacental,300 mg,300.0,MG,Capsule,NaN


In [8]:
df.dose_amt.dtype == "float64"

True

In [7]:
def cleanCol(df, col):
    for i in col: #['drugname','route','dose_vbm', "dose_amt","dose_unit","dose_form","dose_freq"]:
        if df[i].dtype == "float64":
            df[i] = df[i].astype(str)
        else:
            df[i] = np.where((df[i].isnull()) | (df[i] == "UNK"), "nan", df[i].str.lower())

    return df

In [10]:
df = cleanCol(df, ['drugname','route','dose_vbm','dose_amt', 'dose_unit','dose_form','dose_freq'])

In [11]:
df.head()

,primaryid,caseid,drugname,route,dose_vbm,dose_amt,dose_unit,dose_form,dose_freq
0,100144838,10014483,isentress,transplacental,"400 mg, bid",400.0,mg,tablet,bid
1,100144838,10014483,isentress,transplacental,400 mg,400.0,mg,tablet,nan
2,100144838,10014483,isentress,transplacental,nan,nan,nan,tablet,nan
3,100144838,10014483,abacavir,transplacental,"300 mg, bid",300.0,mg,capsule,bid
4,100144838,10014483,abacavir,transplacental,300 mg,300.0,mg,capsule,nan


In [12]:
df['dose'] = np.where(df.dose_vbm != 'nan', df.dose_vbm, np.where(df.dose_amt != 'nan', \
    df[['dose_amt', 'dose_unit','dose_form','dose_freq']].apply(lambda x: " ".join(x.astype(str)), axis=1), "nan"))

In [13]:
df.tail()

,primaryid,caseid,drugname,route,dose_vbm,dose_amt,dose_unit,dose_form,dose_freq,dose
1920727,998834884,9988348,tramadol hydrochloride,nan,nan,nan,nan,nan,nan,nan
1920728,998834884,9988348,voltaren,nan,nan,nan,nan,nan,nan,nan
1920729,998834884,9988348,naproxen,oral,nan,500.0,mg,nan,qd,500.0 mg nan qd
1920730,998834884,9988348,prednisone,nan,nan,nan,nan,nan,nan,nan
1920731,998834884,9988348,methotrexate,nan,nan,nan,nan,nan,nan,nan


In [56]:
treatment = [re.sub(r"[|+|\\+]", "", json.dumps(i.to_dict())) for _, i in df[:20].iterrows()]

In [57]:
len(treatment)

20

In [58]:
treatment[10]

'{"primaryid": 100144838, "caseid": 10014483, "drugname": "bactrim", "route": "transplacental", "dose_vbm": "400 mg, bid", "dose_amt": "400.0", "dose_unit": "mg", "dose_form": "nan", "dose_freq": "bid", "dose": "400 mg, bid"}'

In [28]:
drug_df = pd.read_csv("faers/2025q1/ASCII/DRUG25Q1.txt", sep = "$", low_memory=False)

In [29]:
drug_df.head()

,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai,val_vbm,route,dose_vbm,cum_dose_chr,cum_dose_unit,dechal,rechal,lot_num,exp_dt,nda_num,dose_amt,dose_unit,dose_form,dose_freq
0,100294532,10029453,1,PS,LETROZOLE,LETROZOLE,1,Unknown,UNK,NaN,NaN,U,NaN,NaN,NaN,20726.0,NaN,NaN,NaN,NaN
1,100294532,10029453,2,SS,LAPATINIB,LAPATINIB,1,Unknown,UNK,NaN,NaN,U,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100294532,10029453,3,SS,FULVESTRANT,FULVESTRANT,1,Unknown,UNK,NaN,NaN,U,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100294532,10029453,4,SS,CAPECITABINE,CAPECITABINE,1,Unknown,UNK,NaN,NaN,U,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100294532,10029453,5,SS,TRASTUZUMAB,TRASTUZUMAB,1,Unknown,UNK,NaN,NaN,U,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# define FARES merge func

In [1]:
def mergeFAERS(path_in, path_out):

    '''for every unique 'caseid', only the last 'primaryid' will be used'''
    
    # merge all files(DEMO, DRUG, REAC, OUTC, INDI) in path_in
    for filename in os.listdir(path_in):

        file_out = path_in.split("/")[2]
        
        if "DEMO" in filename.upper() and "TXT" in filename.upper():
            try:
                demo_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)
            except:
                demo_df = pd.read_csv(path_in + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)   
                
            if "sex" in demo_df.columns:
                demo_df.rename(columns = {"sex":'gndr_cod'}, inplace = True)
            #demo_df = demo_df[['primaryid','caseid','age','age_cod','gndr_cod','wt','wt_cod','occp_cod','reporter_country','occr_country']]
            demo_df = demo_df[['fda_dt','rept_cod', 'primaryid','caseid','age','age_cod','gndr_cod','wt','wt_cod']]
            demo_df = demo_df[(demo_df.wt.isnull() == False) & (demo_df.age.isnull() == False)]
            #demo_df.drop('caseid', axis = 1, inplace = True)
            
        if "DRUG" in filename.upper() and "TXT" in filename.upper():
            try:
                drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)
            except:
                drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)   
            
            drug_df = drug_df[['primaryid','caseid', 'drugname','route','dose_vbm','dose_amt',
                               'dose_unit','dose_form','dose_freq']]

            #lower col values
            drug_df = cleanCol(drug_df, ['drugname','route','dose_vbm','dose_amt', 'dose_unit','dose_form','dose_freq'])
            drug_df = drug_df[(drug_df.drugname != "nan") & (drug_df.drugname != "unk")]      

            drug_df['dose'] = np.where(drug_df.dose_vbm != 'nan', drug_df.dose_vbm, np.where(drug_df.dose_amt != 'nan', \
               drug_df[['dose_amt', 'dose_unit','dose_form','dose_freq']].apply(lambda x: " ".join(x.astype(str)), axis=1), "nan"))
            
            drug_df['treatment'] = [re.sub(r"[|+|\\+]", "", json.dumps(i.to_dict())) for _, i in
                      drug_df[['drugname','route','dose']].iterrows()]
            #group by each report and separate by "; "
            drug_df = drug_df.groupby(['primaryid', 'caseid']).treatment.apply(lambda x: "; ".join(x.astype(str))).reset_index()
            
        if "OUTC" in filename.upper() and "TXT" in filename.upper():            
            outc_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)                   
            outc_df = outc_df.groupby(['primaryid', 'caseid'])[outc_df.columns[2]].apply("; ".join).reset_index()
            #outc_df = outc_df.groupby(['primaryid', 'caseid'])[outc_df.columns[2]].last()

        if "INDI" in filename.upper() and "TXT" in filename.upper():            
            indi_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False) 
            indi_df = indi_df.groupby(['primaryid', 'caseid']).indi_pt.apply(lambda x: "; ".join(x.astype(str))).reset_index()
            #indi_df = indi_df.groupby(['primaryid', 'caseid']).indi_pt.last()
            
        if "REAC" in filename.upper() and "TXT" in filename.upper():            
            reac_df = pd.read_csv(path_in + "/" + filename, sep = "$",low_memory=False)            
            reac_df = reac_df.groupby(['primaryid', 'caseid']).pt.apply(lambda x: "; ".join(x.astype(str))).reset_index()
            #reac_df = reac_df.groupby(['primaryid', 'caseid']).pt.last()

    #merge files based on primary report id and case id
    quarter_df = pd.merge(demo_df, drug_df[['primaryid', 'caseid', 'treatment']], 
                          on=['primaryid', 'caseid'], how='inner')  
    quarter_df = pd.merge(quarter_df, outc_df, on=['primaryid', 'caseid'], how = "left")  
    quarter_df = pd.merge(quarter_df, indi_df, on=['primaryid', 'caseid']) # how='inner'
    quarter_df = pd.merge(quarter_df, reac_df, on=['primaryid', 'caseid']) # how='inner'
        
    print(f"The shape of {file_out} is {quarter_df.shape}")

    if os.path.exists(path_out):
        pickle.dump(quarter_df, open(f"{path_out}/{file_out}.pkl", "wb"))
    else:
        !mkdir {path_out}
        pickle.dump(quarter_df, open(f"{path_out}/{file_out}.pkl", "wb"))    
    #pickle.dump(quarter_df, open(f"./aers_merge/{file_out}.pkl", "wb"))

In [3]:
import os, sys 
from functools import partial
from itertools import repeat
from multiprocessing import Pool, freeze_support

In [9]:
oot_lst = ["./faers/" + i + "/ASCII" for i in os.listdir("./faers")]
oot_lst

['./faers/2025q3/ASCII',
 './faers/2025q1/ASCII',
 './faers/2025q2/ASCII',
 './faers/2024q4/ASCII']

In [1]:
# %%time 

# with Pool() as pool:
    
#     pool.map(partial(mergeFAERS, path_out= "./faers_oot"), oot_lst) 

# pool.close() #34K to 70K

# in-parallel prep quarterly data

In [62]:
def mergeAERS(path_in, path_out):

    '''group by unique 'ISR' '''
    #print(f"Processing {path_in}\n")
    # merge all files(DEMO, DRUG, REAC, OUTC, INDI) in path_in
    for filename in os.listdir(path_in):

        file_out = path_in.split("/")[2]
        
        if "DEMO" in filename.upper() and "TXT" in filename.upper():
            try:
                demo_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)
            except:
                demo_df = pd.read_csv(path_in + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)   
            
            demo_df = shiftCol(demo_df)
            if "SEX" in demo_df.columns:
                demo_df.rename(columns = {"SEX":'GNDR_COD'}, inplace = True)
            #demo_df = demo_df[['primaryid','caseid','age','age_cod','gndr_cod','wt','wt_cod','occp_cod','reporter_country','occr_country']]
            demo_df = demo_df[['FDA_DT','REPT_COD',"ISR",'AGE','AGE_COD','GNDR_COD','WT','WT_COD']]
            demo_df = demo_df[(demo_df.WT.isnull() == False) & (demo_df.AGE.isnull() == False)]
            #demo_df.drop('caseid', axis = 1, inplace = True)
            
        if "DRUG" in filename.upper() and "TXT" in filename.upper():
            try:
                drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)
            except:
                drug_df = pd.read_csv(path_in + "/" + filename, sep = "$", encoding='iso-8859-1', low_memory=False)   

            drug_df = shiftCol(drug_df)            
            drug_df.rename(columns = {"DOSE_VBM":"DOSE"}, inplace = True)
            drug_df = drug_df[['ISR', 'DRUGNAME','ROUTE','DOSE']] #subset drug_df
            drug_df = drug_df.loc[(drug_df.DRUGNAME.isnull() == False) & (drug_df.DRUGNAME != "nan"),:] #filter drug_df
            #create 'trestment" col with json dumps 
            drug_df['TREATMENT'] = [re.sub(r"[|+|\\+]", "", json.dumps(i.to_dict())) for _, i 
                                    in drug_df[['DRUGNAME','ROUTE','DOSE']].iterrows()]
            #group by each report and separate by "; "
            drug_df = drug_df.groupby(['ISR']).TREATMENT.apply(lambda x: "; ".join(x.astype(str))).reset_index()
            
        if "OUTC" in filename.upper() and "TXT" in filename.upper():            
            outc_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False)    
            outc_df = shiftCol(outc_df)
            outc_df = outc_df.groupby(['ISR'])[outc_df.columns[1]].apply("; ".join).reset_index()
            #outc_df = outc_df.groupby(['primaryid', 'caseid'])[outc_df.columns[2]].last()

        if "INDI" in filename.upper() and "TXT" in filename.upper():            
            indi_df = pd.read_csv(path_in + "/" + filename, sep = "$", low_memory=False) 
            #indi_df = shiftCol(indi_df)
            indi_df = indi_df.groupby(['ISR']).INDI_PT.apply(lambda x: "; ".join(x.astype(str))).reset_index()
            #indi_df = indi_df.groupby(['primaryid', 'caseid']).indi_pt.last()
            
        if "REAC" in filename.upper() and "TXT" in filename.upper():            
            reac_df = pd.read_csv(path_in + "/" + filename, sep = "$",low_memory=False)      
            reac_df = shiftCol(reac_df)
            reac_df = reac_df.groupby(['ISR']).PT.apply(lambda x: "; ".join(x.astype(str))).reset_index()
            #reac_df = reac_df.groupby(['primaryid', 'caseid']).pt.last()

    #merge files based on primary report id and case id
    quarter_df = pd.merge(demo_df, drug_df[['ISR', 'TREATMENT']], on=['ISR'], how='inner')  
    #print(quarter_df.shape)
    quarter_df = pd.merge(quarter_df, outc_df, on=['ISR'], how = "left")  
    #print(quarter_df.shape)
    quarter_df = pd.merge(quarter_df, indi_df, on=['ISR']) # how='inner'
    #print(quarter_df.shape)
    quarter_df = pd.merge(quarter_df, reac_df, on=['ISR']) # how='inner'
        
    print(f"The shape of {file_out} is {quarter_df.shape}")

    if os.path.exists(path_out):
        pickle.dump(quarter_df, open(f"{path_out}/{file_out}.pkl", "wb"))
    else:
        !mkdir {path_out}
        pickle.dump(quarter_df, open(f"{path_out}/{file_out}.pkl", "wb"))  

In [2]:
# %%time 

# with Pool() as pool:    
#     pool.map(partial(mergeAERS, path_out= "./aers_all"), aers_in) 

# pool.close() #17K to 38K

# consolidate aers pkls into a single pickle

In [ ]:
aers_lst = os.listdir('../aers_all/')
aers_lst = sorted(aers_lst)
aers_lst[-1]

'2012q3.pkl'

In [ ]:
%%time 

aers_df = pd.DataFrame()
for i in tqdm(aers_lst):
    pkl_i = pickle.load(open(f"../aers_all/{i}", "rb"))
    if "OUTC_COD" in pkl_i.columns:        
        '''rename outc_cod to outc_code before appending'''
        pkl_i.rename(columns = {"OUTC_COD": "OUTC_CODE"}, inplace = True)    
    
    pkl_i["yr_qtr"] = i[:6]        
    aers_df = pd.concat([aers_df, pkl_i])

100%|██████████| 35/35 [00:01<00:00, 18.25it/s]

CPU times: user 1.78 s, sys: 145 ms, total: 1.92 s
Wall time: 1.97 s


In [135]:
aers_df.shape

(1006101, 13)

In [136]:
aers_df.head()

,FDA_DT,REPT_COD,ISR,AGE,AGE_COD,GNDR_COD,WT,WT_COD,TREATMENT,OUTC_CODE,INDI_PT,PT,yr_qtr
0,20031229,EXP,4261678,52.0,YR,F,200.0,LBS,"{""DRUGNAME"": ""ZITHROMAX"", ""ROUTE"": ""ORAL"", ""DO...",HO; OT,SKIN DISORDER; BENIGN INTRACRANIAL HYPERTENSIO...,BRONCHITIS; DIARRHOEA; DISORIENTATION; DRUG IN...,2004q1
1,20040102,EXP,4261826,71.0,YR,M,110.0,KG,"{""DRUGNAME"": ""TAXOTERE"", ""ROUTE"": NaN, ""DOSE"":...",HO,OESOPHAGEAL CARCINOMA; HYPERTENSION; ATRIAL FI...,CUTANEOUS VASCULITIS; FEBRILE NEUTROPENIA; REN...,2004q1
2,20040102,EXP,4261829,51.0,YR,F,82.1,KG,"{""DRUGNAME"": ""DOCETAXEL"", ""ROUTE"": ""INTRAVENOU...",DE; HO,ANAL CANCER; ANAL CANCER; PAIN,ANAL CANCER; COAGULOPATHY; DYSPNOEA; GASTROINT...,2004q1
3,20040102,EXP,4261860,49.0,YR,M,35.0,KG,"{""DRUGNAME"": ""SIMULECT"", ""ROUTE"": ""INTRAVENOUS...",DE; HO,RENAL TRANSPLANT; RENAL TRANSPLANT; RENAL TRAN...,ACUTE PULMONARY OEDEMA; ACUTE RESPIRATORY DIST...,2004q1
4,20040102,EXP,4261872,72.0,YR,M,63.0,KG,"{""DRUGNAME"": ""DIOVAN"", ""ROUTE"": ""ORAL"", ""DOSE""...",DS; HO,HYPERTENSION; AGE INDETERMINATE MYOCARDIAL INF...,ATRIOVENTRICULAR BLOCK; ATRIOVENTRICULAR BLOCK...,2004q1


In [10]:
pickle.dump(aers_df, open("aers_df.pkl", "wb"))

In [10]:
def onlyCode(code:str):
    return np.where("DE" in code, "DE", 
            np.where("LT" in code, "LT", 
             np.where("HO" in code, "HO",
              np.where("DS" in code, "DS", 
                np.where("CA" in code, "CA",
                 np.where("RI" in code, "RI",
                  np.where("OT" in code, "OT", "NA")))))))

In [141]:
aers_df.OUTC_CODE[:5]

0    HO; OT
1        HO
2    DE; HO
3    DE; HO
4    DS; HO
Name: OUTC_CODE, dtype: str

In [142]:
import numpy as np
aers_df.OUTC_CODE = aers_df.OUTC_CODE.astype(str)
aers_df['uni_code'] = [onlyCode(str(i)) for i in aers_df.OUTC_CODE]

In [143]:
aers_df.columns = [i.lower() for i in aers_df.columns] #lower case col names

In [145]:
aers_df.head(2)

,fda_dt,rept_cod,isr,age,age_cod,gndr_cod,wt,wt_cod,treatment,outc_code,indi_pt,pt,yr_qtr,uni_code
0,20031229,EXP,4261678,52.0,YR,F,200.0,LBS,"{""DRUGNAME"": ""ZITHROMAX"", ""ROUTE"": ""ORAL"", ""DO...",HO; OT,SKIN DISORDER; BENIGN INTRACRANIAL HYPERTENSIO...,BRONCHITIS; DIARRHOEA; DISORIENTATION; DRUG IN...,2004q1,HO
1,20040102,EXP,4261826,71.0,YR,M,110.0,KG,"{""DRUGNAME"": ""TAXOTERE"", ""ROUTE"": NaN, ""DOSE"":...",HO,OESOPHAGEAL CARCINOMA; HYPERTENSION; ATRIAL FI...,CUTANEOUS VASCULITIS; FEBRILE NEUTROPENIA; REN...,2004q1,HO


# create ARES `inst` string

In [146]:
%%time

aers_df['patient'] = [re.sub(r"[|+|\\+]", "", json.dumps(row.to_dict())) 
                        for _, row in aers_df[['age','age_cod','gndr_cod','wt','wt_cod']].astype(str).iterrows()]
aers_df.drop(['age','age_cod','gndr_cod','wt','wt_cod'], axis = 1, inplace = True)

aers_df.pt = [i.lower() for i in aers_df.pt]

CPU times: user 1min 28s, sys: 177 ms, total: 1min 29s
Wall time: 1min 29s


# prepare aers triplet

In [15]:
def prepTriplet(df):
    _jsonl = df[['patient','treatment','indi_pt']].astype(str).to_json(orient='records', lines = True)
    # split jsonl except 1
    _jsonl = _jsonl.split("\n{")

    #reformate jsonl
    _jsonl[1:] = ["{" + _jsonl[i] for i in range(1, len(_jsonl))]

    _jsonl = [re.sub(r"[|+|\\+]", "", i) for i in _jsonl]
    return _jsonl   

In [148]:
aers_df['inst'] = prepTriplet(aers_df[['patient','treatment','indi_pt']])
aers_df.drop(['patient','treatment','indi_pt'], axis = 1, inplace = True)

In [149]:
aers_df.head()

,fda_dt,rept_cod,isr,outc_code,pt,yr_qtr,uni_code,inst
0,20031229,EXP,4261678,HO; OT,bronchitis; diarrhoea; disorientation; drug in...,2004q1,HO,"{""patient"":""{""age"": ""52.0"", ""age_cod"": ""YR"", ""..."
1,20040102,EXP,4261826,HO,cutaneous vasculitis; febrile neutropenia; ren...,2004q1,HO,"{""patient"":""{""age"": ""71.0"", ""age_cod"": ""YR"", ""..."
2,20040102,EXP,4261829,DE; HO,anal cancer; coagulopathy; dyspnoea; gastroint...,2004q1,DE,"{""patient"":""{""age"": ""51.0"", ""age_cod"": ""YR"", ""..."
3,20040102,EXP,4261860,DE; HO,acute pulmonary oedema; acute respiratory dist...,2004q1,DE,"{""patient"":""{""age"": ""49.0"", ""age_cod"": ""YR"", ""..."
4,20040102,EXP,4261872,DS; HO,atrioventricular block; atrioventricular block...,2004q1,HO,"{""patient"":""{""age"": ""72.0"", ""age_cod"": ""YR"", ""..."


In [219]:
aers_df.reset_index(drop = True, inplace = True)

In [151]:
aers_df.inst = [i.lower() for i in aers_df.inst]

In [152]:
aers_df.pt = [i.lower() for i in aers_df.pt]

In [153]:
aers_df.head(2)

,fda_dt,rept_cod,isr,outc_code,pt,yr_qtr,uni_code,inst
0,20031229,EXP,4261678,HO; OT,bronchitis; diarrhoea; disorientation; drug in...,2004q1,HO,"{""patient"":""{""age"": ""52.0"", ""age_cod"": ""yr"", ""..."
1,20040102,EXP,4261826,HO,cutaneous vasculitis; febrile neutropenia; ren...,2004q1,HO,"{""patient"":""{""age"": ""71.0"", ""age_cod"": ""yr"", ""..."


In [224]:
pickle.dump(aers_df, open("../adr_up2_2012_q3.pkl", "wb"))

In [3]:
aers_df = pickle.load(open("../adr_up2_2012_q3.pkl", "rb"))

In [11]:
aers_df.shape #1M

(1006101, 8)

In [334]:
sorted(aers_df.yr_qtr.unique())[-5:]

['2011q3', '2011q4', '2012q1', '2012q2', '2012q3']

In [2]:
import os

In [4]:
pkl_lst = os.listdir('../faers_all/')
pkl_lst = sorted(pkl_lst)
# pkl_lst[-1]

In [157]:
pkl_lst[-3:]

['2025q1.pkl', '2025q2.pkl', '2025q3.pkl']

# prepare FAERS data for hybrid RAG ACL model

In [17]:
pkl_lst[:-1][-1]

'2025q2.pkl'

In [5]:
%%time 

append_df = pd.DataFrame()
for i in tqdm(pkl_lst[:-1]):
    pkl_i = pickle.load(open(f"../faers_all/{i}", "rb"))
    if "outc_cod" in pkl_i.columns:        
        '''rename outc_cod to outc_code before appending'''
        pkl_i.rename(columns = {"outc_cod": "outc_code"}, inplace = True)   
    
    pkl_i["yr_qtr"] = i[:6]        
    append_df = pd.concat([append_df, pkl_i])

100%|██████████| 51/51 [00:08<00:00,  5.69it/s]

CPU times: user 7.78 s, sys: 1.2 s, total: 8.98 s
Wall time: 8.97 s


In [338]:
sorted(append_df.yr_qtr.unique())[-5:]

['2024q2', '2024q3', '2024q4', '2025q1', '2025q2']

In [8]:
import numpy as np
append_df.outc_code = append_df.outc_code.fillna("NA").astype(str)
append_df['uni_code'] = [onlyCode(i) for i in append_df.outc_code]

# combine faers and aers df

In [340]:
append_df.shape

(2804382, 15)

# create FARES inst string

In [10]:
%%time

append_df['patient'] = [re.sub(r"[|+|\\+]", "", json.dumps(row.to_dict()))
                        for _, row in append_df[['age','age_cod','gndr_cod','wt','wt_cod']].astype(str).iterrows()]
append_df.drop(['age','age_cod','gndr_cod','wt','wt_cod'], axis = 1, inplace = True)
append_df['inst'] = prepTriplet(append_df[['patient','treatment','indi_pt']])
append_df.drop(['patient','treatment','indi_pt'], axis = 1, inplace = True)
append_df.pt = [i.lower() for i in append_df.pt]
append_df.inst = [i.lower() for i in append_df.inst]

CPU times: user 4min 50s, sys: 17.7 s, total: 5min 8s
Wall time: 5min 8s


In [ ]:
#append_df.reset_index(inplace = True, drop = True)

In [17]:
append_df.columns

Index(['fda_dt', 'rept_cod', 'primaryid', 'caseid', 'outc_code', 'pt',
       'yr_qtr', 'uni_code', 'inst'],
      dtype='object')

In [35]:
# #pickle.dump(append_df, open("adr_up2_2025_q2.pkl", "wb"))
append_df = pickle.load(open("adr_up2_2025_q1.pkl", "rb"))

In [36]:
append_df.shape

(2749372, 9)

# make sure no caseid overlapped 

In [19]:
aers_df = pickle.load(open("../ADR_KG/aers_kg.pkl", "rb"))
aers_df.head()

,fda_dt,rept_cod,isr,age,age_cod,gndr_cod,wt,wt_cod,drugname,outc_code,indi_pt,pt,yr_qtr,uni_code
0,20031229,EXP,4261678,52.0,YR,F,200.0,LBS,ACETAZOLAMIDE; ANALGESICS; ZITHROMAX,HO; OT,BENIGN INTRACRANIAL HYPERTENSION; HEADACHE; SK...,BRONCHITIS; DIARRHOEA; DISORIENTATION; DRUG IN...,2004q1,HO
1,20040102,EXP,4261826,71.0,YR,M,110.0,KG,BETA BLOCKING AGENTS; BETATOP; MOPRAL; PREVISC...,HO,ATRIAL FIBRILLATION; HYPERTENSION; OESOPHAGEAL...,CUTANEOUS VASCULITIS; FEBRILE NEUTROPENIA; REN...,2004q1,HO
2,20040102,EXP,4261829,51.0,YR,F,82.1,KG,ANALGESIC LIQ; COUMADIN; DOCETAXEL; PS-341,DE; HO,ANAL CANCER; ANAL CANCER; PAIN,ANAL CANCER; COAGULOPATHY; DYSPNOEA; GASTROINT...,2004q1,DE
3,20040102,EXP,4261860,49.0,YR,M,35.0,KG,ACETAMINOPHEN; ALOSENN; AMOBAN; ASPIRIN; BAKTA...,DE; HO,RENAL TRANSPLANT; RENAL TRANSPLANT; RENAL TRAN...,ACUTE PULMONARY OEDEMA; ACUTE RESPIRATORY DIST...,2004q1,DE
4,20040102,EXP,4261872,72.0,YR,M,63.0,KG,BUFFERIN; DIOVAN; MARZULENE; MEVALOTIN; NORVASC,DS; HO,AGE INDETERMINATE MYOCARDIAL INFARCTION; GASTR...,ATRIOVENTRICULAR BLOCK; ATRIOVENTRICULAR BLOCK...,2004q1,HO


In [20]:
aers_df.rename(columns = {"isr":"caseid"}, inplace = True)

In [21]:
aers_df['primaryid'] = aers_df.caseid

In [22]:
append_df.columns

Index(['fda_dt', 'rept_cod', 'primaryid', 'caseid', 'outc_code', 'pt',
       'yr_qtr', 'uni_code', 'inst'],
      dtype='object')

In [23]:
aers_df.columns

Index(['fda_dt', 'rept_cod', 'caseid', 'age', 'age_cod', 'gndr_cod', 'wt',
       'wt_cod', 'drugname', 'outc_code', 'indi_pt', 'pt', 'yr_qtr',
       'uni_code', 'primaryid'],
      dtype='str')

In [24]:
col_in = ["fda_dt","rept_cod","primaryid","caseid","outc_code","pt","yr_qtr"]

In [37]:
adr_all = pd.concat([append_df[col_in], aers_df[col_in]])

In [38]:
adr_all.shape #3755473

(3755473, 7)

In [39]:
adr_all = adr_all.sort_values(["yr_qtr", "fda_dt"])

In [41]:
# import old adr data set up to 2025q1

In [43]:
adr_dedup = pickle.load(open("adr_all_7_27_2025.pkl", "rb"))

In [44]:
adr_trn = adr_dedup.loc[adr_dedup.yr_qtr < "2024q4"]

In [45]:
adr_trn.shape

(3387476, 9)

In [46]:
adr_trn.head()

,fda_dt,rept_cod,caseid,outc_code,pt,yr_qtr,uni_code,inst,outcome
0,20121120,EXP,37831703,HO,methaemoglobinaemia; overdose,2012q4,HO,"{""patient"":""{""age"": ""3"", ""age_cod"": ""yr"", ""gnd...","{""pt"": ""methaemoglobinaemia; overdose"", ""uni_c..."
1,20120920,EXP,37883263,OT,hot flush; thermal burn,2012q4,OT,"{""patient"":""{""age"": ""49"", ""age_cod"": ""yr"", ""gn...","{""pt"": ""hot flush; thermal burn"", ""uni_code"": ..."
2,20121226,EXP,38941784,HO,acute tonsillitis; anaemia; cellulitis gangren...,2012q4,HO,"{""patient"":""{""age"": ""50"", ""age_cod"": ""yr"", ""gn...","{""pt"": ""acute tonsillitis; anaemia; cellulitis..."
3,20120906,EXP,39422278,OT; HO,brain natriuretic peptide increased; c-reactiv...,2012q4,HO,"{""patient"":""{""age"": ""66"", ""age_cod"": ""yr"", ""gn...","{""pt"": ""brain natriuretic peptide increased; c..."
4,20121004,EXP,40412084,HO; LT,abscess; cardiac arrest; cerebral atrophy; der...,2012q4,LT,"{""patient"":""{""age"": ""8"", ""age_cod"": ""mon"", ""gn...","{""pt"": ""abscess; cardiac arrest; cerebral atro..."


In [47]:
pickle.dump(adr_trn, open("adr_trn_2024q3.pkl", "wb"))

In [23]:
adr_key = pickle.load(open("hybrid_pt_key.pkl", "rb"))

In [25]:
adr_all_['key'] = adr_all_.pt.str[:10] + "|||" + adr_all_.pt.str[-10:]

In [61]:
# get training set that adr_all_.yr_qtr <= "2024q3, which means the most recent quarter is 2024q2
adr_trn = adr_all_[adr_all_.yr_qtr <= "2024q3"]

In [62]:
adr_trn.shape

(3640883, 8)

In [ ]:
#adr_trn = pd.merge(adr_all, adr_key, on='key', how='inner')

In [436]:
# # check duplicate primaryid
# adr_all.primaryid.duplicated().sum()

In [ ]:
# # sort fda_dt by ascending order, drop duplicates primaryid
# adr_all = adr_all.sort_values("fda_dt").drop_duplicates(subset = "primaryid", keep = "last")

In [399]:
import json

In [63]:
# check duplicated inst
adr_trn.inst.duplicated().sum()

np.int64(248601)

In [64]:
adr_trn.head()

,fda_dt,rept_cod,primaryid,outc_code,pt,yr_qtr,uni_code,inst
0,20031229,EXP,4261678,HO; OT,bronchitis; diarrhoea; disorientation; drug in...,2004q1,HO,"{""patient"":""{""age"": ""52.0"", ""age_cod"": ""yr"", ""..."
1,20040102,EXP,4261826,HO,cutaneous vasculitis; febrile neutropenia; ren...,2004q1,HO,"{""patient"":""{""age"": ""71.0"", ""age_cod"": ""yr"", ""..."
2,20040102,EXP,4261829,DE; HO,anal cancer; coagulopathy; dyspnoea; gastroint...,2004q1,DE,"{""patient"":""{""age"": ""51.0"", ""age_cod"": ""yr"", ""..."
3,20040102,EXP,4261860,DE; HO,acute pulmonary oedema; acute respiratory dist...,2004q1,DE,"{""patient"":""{""age"": ""49.0"", ""age_cod"": ""yr"", ""..."
4,20040102,EXP,4261872,DS; HO,atrioventricular block; atrioventricular block...,2004q1,HO,"{""patient"":""{""age"": ""72.0"", ""age_cod"": ""yr"", ""..."


In [65]:
# drop duplicated inst and keep the last based on fda_dt except oot
adr_trn = adr_trn.sort_values("fda_dt").drop_duplicates(subset = "inst", keep = "last")

In [69]:
adr_trn.shape

(3391573, 8)

In [68]:
# drop duplicate primaryid
adr_trn = adr_trn.drop_duplicates(subset = "primaryid", keep = "last")

In [70]:
#adr_all_.drop(['fda_dt','rept_cod'], axis = 1, inplace = True)

In [71]:
# adr_all_["outcome"] = [re.sub(r"[|+|\\+]", "", json.dumps(row.to_dict()))
#                         for _, row in adr_all_[["pt","uni_code"]].astype(str).iterrows()]

# pull merged adr of all up to 2024q2 #(3755473, 8)

In [428]:
# check shape of adr_train data (2004q1-2024q2)
adr_trn = adr_all[adr_all.yr_qtr.isin(['2024q4', '2025q1', '2025q2']) == False]

In [429]:
# adr training set covers up to 2024q3 !!!
print(adr_trn.shape)
adr_trn.head() #(3387476, 5)

(3386520, 7)


,primaryid,outc_code,pt,yr_qtr,uni_code,inst,outcome
2709754,31231172,DE; HO,arterial thrombosis; drug interaction; haemorr...,2020q3,DE,"{""patient"":""{""age"": ""59.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""arterial thrombosis; drug interaction;..."
2709758,32427281,OT,anaemia; dermatitis; ecchymosis,2020q3,OT,"{""patient"":""{""age"": ""46.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""anaemia; dermatitis; ecchymosis"", ""uni..."
2709773,37847331,RI,blood pressure decreased; rash erythematous,2020q3,RI,"{""patient"":""{""age"": ""72.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""blood pressure decreased; rash erythem..."
2709755,32030921,LT; DE,blister; dermatitis; lip disorder; mucosal ero...,2020q3,DE,"{""patient"":""{""age"": ""34.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""blister; dermatitis; lip disorder; muc..."
2709756,32138152,LT; DE,conjunctivitis; mouth ulceration; oral mucosal...,2020q3,DE,"{""patient"":""{""age"": ""79.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""conjunctivitis; mouth ulceration; oral..."


In [415]:
1612 + 3385811

3387423

In [ ]:
#pickle.dump(adr_dedup.loc[adr_dedup.yr_qtr < "2024q3"], open("adr_train.pkl", "wb")) #"adr_train_old.pkl"

In [ ]:
# import pickle
# adr_trn = pickle.load(open("adr_train.pkl", "rb"))

In [67]:
adr_trn.yr_qtr.unique().max()

'2024q2'

# recreate OOT from q4'24 to q2'25 

In [45]:
import os, sys

In [46]:
pkl_lst = os.listdir('../faers_all/')
pkl_lst = sorted(pkl_lst)
pkl_lst[-4:-1]

['2024q4.pkl', '2025q1.pkl', '2025q2.pkl']

In [47]:
%%time 

oot_df = pd.DataFrame()
for i in tqdm(pkl_lst[-4:-1]):
    pkl_i = pickle.load(open(f"../faers_all/{i}", "rb"))
    if "outc_cod" in pkl_i.columns:        
        '''rename outc_cod to outc_code before appending'''
        pkl_i.rename(columns = {"outc_cod": "outc_code"}, inplace = True)   
    
    pkl_i["yr_qtr"] = i[:6]        
    oot_df = pd.concat([oot_df, pkl_i])

100%|██████████| 3/3 [00:00<00:00, 10.50it/s]

CPU times: user 235 ms, sys: 66.7 ms, total: 301 ms
Wall time: 346 ms


In [57]:
oot_df.shape

(169600, 14)

In [56]:
pickle.dump(oot_df[['primaryid','outc_code']], open("adr_oot_outc.pkl", "wb"))

In [11]:
oot_df.outc_code = oot_df.outc_code.fillna("NA").astype(str)
oot_df['uni_code'] = [onlyCode(i) for i in oot_df.outc_code]

In [13]:
oot_df.columns

Index(['fda_dt', 'rept_cod', 'primaryid', 'caseid', 'treatment', 'outc_code',
       'indi_pt', 'pt', 'yr_qtr', 'uni_code', 'patient'],
      dtype='str')

In [12]:
%%time

oot_df['patient'] = [re.sub(r"[|+|\\+]", "", json.dumps(row.to_dict()))
                        for _, row in oot_df[['age','age_cod','gndr_cod','wt','wt_cod']].astype(str).iterrows()]
oot_df.drop(['age','age_cod','gndr_cod','wt','wt_cod'], axis = 1, inplace = True)

CPU times: user 15.5 s, sys: 94.7 ms, total: 15.6 s
Wall time: 15.6 s


In [16]:
oot_df['inst'] = prepTriplet(oot_df[['patient','treatment','indi_pt']])
oot_df.drop(['patient','treatment','indi_pt'], axis = 1, inplace = True)
oot_df.pt = [i.lower() for i in oot_df.pt]
oot_df.inst = [i.lower() for i in oot_df.inst]

In [35]:
oot_df = oot_df.sort_values(["fda_dt"])
oot_dedup = oot_df.drop_duplicates(subset = "inst", keep='last') #drop both same caseid and same inst string

In [36]:
oot_dedup["outcome"] = [re.sub(r"[|+|\\+]", "", json.dumps(row.to_dict()))
                        for _, row in oot_dedup[["pt","uni_code"]].astype(str).iterrows()]

In [37]:
print(oot_dedup.shape) #162458
oot_dedup.head()

(163259, 10)


,fda_dt,rept_cod,primaryid,caseid,outc_code,pt,yr_qtr,uni_code,inst,outcome
10470,20241001,EXP,240445604,24044560,OT; HO,thrombocytopenia; anaemia,2024q4,HO,"{""patient"":""{""age"": ""73.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""thrombocytopenia; anaemia"", ""uni_code""..."
3433,20241001,EXP,213582709,21358270,OT,cardiac failure; off label use; hypoaesthesia;...,2024q4,OT,"{""patient"":""{""age"": ""81.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""cardiac failure; off label use; hypoae..."
3452,20241001,EXP,213735722,21373572,HO,erysipelas; pyrexia,2024q4,HO,"{""patient"":""{""age"": ""60.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""erysipelas; pyrexia"", ""uni_code"": ""HO""}"
3483,20241001,EXP,213952233,21395223,NA,covid-19,2024q4,NA,"{""patient"":""{""age"": ""53.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""covid-19"", ""uni_code"": ""NA""}"
3510,20241001,EXP,214200063,21420006,DE; HO; LT; OT,hypernatraemia; quadriplegia; unresponsive to ...,2024q4,DE,"{""patient"":""{""age"": ""39.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""hypernatraemia; quadriplegia; unrespon..."


In [38]:
oot_dedup.drop("caseid", axis = 1, inplace = True)

In [39]:
oot_dedup.rename(columns = {"primaryid":"caseid"}, inplace = True)

In [40]:
pickle.dump(oot_dedup[["caseid", 'outc_code']], open("adr_oot_outc.pkl", "wb"))

In [41]:
import pickle
oot_dedup_code = pickle.load(open("adr_oot_outc.pkl", "rb"))

In [44]:
"237392373" in oot_dedup_code.caseid

False

# Filter uni_code prediction without NA

In [28]:
oot_no_na = oot_dedup[oot_dedup.uni_code != "NA"]

In [29]:
oot_no_na.shape #(118954, 10) 2 more?

(118952, 9)

In [30]:
oot_no_na.reset_index(inplace = True)

In [31]:
oot_no_na.head()

,index,fda_dt,rept_cod,caseid,outc_code,pt,yr_qtr,uni_code,inst,outcome
0,10470,20241001,EXP,240445604,OT; HO,thrombocytopenia; anaemia,2024q4,HO,"{""patient"":""{""age"": ""73.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""thrombocytopenia; anaemia"", ""uni_code""..."
1,3433,20241001,EXP,213582709,OT,cardiac failure; off label use; hypoaesthesia;...,2024q4,OT,"{""patient"":""{""age"": ""81.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""cardiac failure; off label use; hypoae..."
2,3452,20241001,EXP,213735722,HO,erysipelas; pyrexia,2024q4,HO,"{""patient"":""{""age"": ""60.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""erysipelas; pyrexia"", ""uni_code"": ""HO""}"
3,3510,20241001,EXP,214200063,DE; HO; LT; OT,hypernatraemia; quadriplegia; unresponsive to ...,2024q4,DE,"{""patient"":""{""age"": ""39.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""hypernatraemia; quadriplegia; unrespon..."
4,3509,20241001,EXP,214189118,HO,pyelonephritis; urinary tract infection; aphth...,2024q4,HO,"{""patient"":""{""age"": ""32.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""pyelonephritis; urinary tract infectio..."


In [32]:
len(oot_no_na.caseid.unique()) == len(oot_no_na)

False

In [33]:
oot_no_na = oot_no_na.drop_duplicates("inst", keep = "last")

In [34]:
oot_no_na.shape #hybrid rag oot with no NA code in use

(118952, 10)

# drop oot_no_na caseid in train set

In [67]:
oot_no_na.caseid.isin(adr_trn.caseid).sum()

0

In [68]:
oot_no_na = oot_no_na[oot_no_na.caseid.isin(adr_trn.caseid) == False]

In [69]:
oot_no_na.head()

,level_0,index,fda_dt,rept_cod,caseid,outc_code,pt,yr_qtr,uni_code,inst,outcome
0,0,16258,20241001,EXP,243847721,HO; OT,genital swelling; near death experience; weigh...,2024q4,HO,"{""patient"":""{""age"": ""55.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""genital swelling; near death experienc..."
1,1,8432,20241001,EXP,237436793,HO; OT,drug ineffective; abdominal pain,2024q4,HO,"{""patient"":""{""age"": ""65.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""drug ineffective; abdominal pain"", ""un..."
2,2,8429,20241001,EXP,237432054,HO; OT,cardiac failure; weight decreased; sepsis; ina...,2024q4,HO,"{""patient"":""{""age"": ""82.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""cardiac failure; weight decreased; sep..."
3,3,16282,20241001,EXP,243849271,HO,inappropriate schedule of product administrati...,2024q4,HO,"{""patient"":""{""age"": ""42.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""inappropriate schedule of product admi..."
4,4,16285,20241001,EXP,243849681,DE,sepsis; febrile neutropenia,2024q4,DE,"{""patient"":""{""age"": ""61.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""sepsis; febrile neutropenia"", ""uni_cod..."


In [ ]:
pickle.dump(oot_no_na[["caseid","pt","yr_qtr","outc_code","inst","outcome"]], open("oot_hybrid.pkl", "wb")) #FINAL!

In [122]:
oot = pickle.load(open("oot_not_na.pkl","rb"))

In [123]:
print(oot.shape)
oot.head(2)

(118953, 6)


,caseid,pt,yr_qtr,uni_code,inst,outcome
0,243847721,genital swelling; near death experience; weigh...,2024q4,HO,"{""patient"":""{""age"": ""55.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""genital swelling; near death experienc..."
1,237436793,drug ineffective; abdominal pain,2024q4,HO,"{""patient"":""{""age"": ""65.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""drug ineffective; abdominal pain"", ""un..."


In [76]:
oot_lbl = pickle.load(open("adr_oot_labels.pkl", "rb"))

In [79]:
oot_lbl[:2]

['Eye disorders', 'Immune system disorders']

In [80]:
oot.pt[0]

'genital swelling; near death experience; weight decreased; dehydration; limb injury; peripheral swelling; urine output decreased; burning sensation'

In [84]:
oot.inst[0]

'{"patient":"{"age": "55.0", "age_cod": "yr", "gndr_cod": "m", "wt": "99.79", "wt_cod": "kg"}","treatment":"{"drugname": "furosemide", "route": "intravenous (not otherwise specified)", "dose": "40 mg"}; {"drugname": "furosemide", "route": "nan", "dose": "taking 3 tablets at at time, for total of 60 mg, once a day."}","indi_pt":"fluid retention"}'

In [81]:
oot_scr = pickle.load(open("adr_oot_scores.pkl", "rb"))

In [82]:
oot_scr[:2]

['Eye disorders', 'Immune system disorders']

# clean training and oot 
* limit the case to those using only PTs in SOC mapping table

In [2]:
pt_soc_map = pickle.load(open("mapping_str.pkl", "rb"))

In [3]:
pt_soc_map.head()

0         Acute abdomen; Gastrointestinal disorders
1        Abdominal mass; Gastrointestinal disorders
2    Abdominal neoplasm; Gastrointestinal disorders
3        Abdominal pain; Gastrointestinal disorders
4              Abetalipoproteinaemia; Eye disorders
dtype: str

In [11]:
# get PT terms from pt_soc_map - MED
pts = [i.split("; ")[0].lower() for i in pt_soc_map]

In [12]:
print(len(pts))
pts[:5]

26912


['acute abdomen',
 'abdominal mass',
 'abdominal neoplasm',
 'abdominal pain',
 'abetalipoproteinaemia']

In [27]:
"off label use" in pts

True

In [6]:
adr = pickle.load(open("./adr_trn_new.pkl", "rb"))
#adr.reset_index(names = ["id"], inplace = True)
print(adr.shape)
#adr.id = adr.id.astype(str)
adr.head()

(3387476, 8)


,id,inst,outcome,uni_code,pt,caseid,yr_qtr,outc_code
0,0,"{""patient"":""{""age"": ""59.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""arterial thrombosis; drug interaction;...",DE,arterial thrombosis; drug interaction; haemorr...,31231172,2020q3,DE; HO
1,1,"{""patient"":""{""age"": ""46.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""anaemia; dermatitis; ecchymosis"", ""uni...",OT,anaemia; dermatitis; ecchymosis,32427281,2020q3,OT
2,2,"{""patient"":""{""age"": ""72.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""blood pressure decreased; rash erythem...",RI,blood pressure decreased; rash erythematous,37847331,2020q3,RI
3,3,"{""patient"":""{""age"": ""34.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""blister; dermatitis; lip disorder; muc...",DE,blister; dermatitis; lip disorder; mucosal ero...,32030921,2020q3,LT; DE
4,4,"{""patient"":""{""age"": ""79.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""conjunctivitis; mouth ulceration; oral...",DE,conjunctivitis; mouth ulceration; oral mucosal...,32138152,2020q3,LT; DE


# check if a split pt in adr are all in pts list

In [15]:
%%time

adr["in_pt"] = [all(term in pts for term in pt.split("; ")) for pt in adr.pt]

CPU times: user 9min 20s, sys: 145 ms, total: 9min 20s
Wall time: 9min 21s


In [ ]:
sum(adr.in_pt)/len(adr) #88.6% are in PT

0.8860369785645714

In [51]:
adr[adr["in_pt"] == False].tail(10)

,id,inst,outcome,uni_code,pt,caseid,yr_qtr,outc_code,in_pt
3385285,3385285,"{""patient"":""{""age"": ""62.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""drug hypersensitivity; blood pressure ...",OT,drug hypersensitivity; blood pressure systolic...,228145943,2024q3,OT,False
3385546,3385546,"{""patient"":""{""age"": ""80.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""dyspnoea; off label use; bronchiectasi...",HO,dyspnoea; off label use; bronchiectasis; mictu...,1997261118,2024q3,CA; DS; HO; OT,False
3386289,3386289,"{""patient"":""{""age"": ""50.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""blood triglycerides abnormal; immune s...",LT,blood triglycerides abnormal; immune system di...,243690321,2024q3,OT; DS; LT,False
3386527,3386527,"{""patient"":""{""age"": ""28.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""exposure during pregnancy; arthralgia;...",HO,exposure during pregnancy; arthralgia; agranul...,2354392311,2024q3,HO; OT,False
3386647,3386647,"{""patient"":""{""age"": ""80.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""neutropenia; weight decreased; disease...",HO,neutropenia; weight decreased; disease progres...,1794326414,2024q3,HO; OT,False
3386731,3386731,"{""patient"":""{""age"": ""80.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""off label use; joint injury; benign pr...",HO,off label use; joint injury; benign prostatic ...,1872426638,2024q3,CA; DS; HO; OT,False
3386822,3386822,"{""patient"":""{""age"": ""60.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""hypertension; spinal compression fract...",HO,hypertension; spinal compression fracture; dys...,2263943914,2024q3,DS; HO; OT,False
3386831,3386831,"{""patient"":""{""age"": ""49.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""venous thrombosis limb; pulmonary mass...",HO,venous thrombosis limb; pulmonary mass; muscle...,2301643615,2024q3,HO; OT,False
3386856,3386856,"{""patient"":""{""age"": ""50.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""chondropathy; synovial cyst; brain inj...",LT,chondropathy; synovial cyst; brain injury; met...,2311163610,2024q3,OT; LT; DS,False
3387354,3387354,"{""patient"":""{""age"": ""10.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""diarrhoea; off label use; lung abscess...",HO,diarrhoea; off label use; lung abscess,243811461,2024q3,HO; OT,False


In [46]:
adr[adr["in_pt"] == False].pt[90]

'cataract; deafness unilateral; hearing impaired; myocardial infarction'

In [53]:
"myocardial infarction" in pts

True

In [35]:
tt = adr[adr["in_pt"] == False].pt[3386731].split("; ")

In [40]:
[i in pts for i in tt].index(False)

13

In [41]:
tt[13]

'malignant melanoma in situ'

In [16]:
oot = pickle.load(open("./adr_oot_new.pkl", "rb"))
oot.reset_index(drop = True, inplace = True)
print(oot.shape)
oot.head()

(118953, 7)


,caseid,pt,yr_qtr,uni_code,inst,outcome,outc_code
0,243847721,genital swelling; near death experience; weigh...,2024q4,HO,"{""patient"":""{""age"": ""55.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""genital swelling; near death experienc...",HO; OT
1,237436793,drug ineffective; abdominal pain,2024q4,HO,"{""patient"":""{""age"": ""65.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""drug ineffective; abdominal pain"", ""un...",HO; OT
2,237432054,cardiac failure; weight decreased; sepsis; ina...,2024q4,HO,"{""patient"":""{""age"": ""82.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""cardiac failure; weight decreased; sep...",HO; OT
3,243849271,inappropriate schedule of product administrati...,2024q4,HO,"{""patient"":""{""age"": ""42.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""inappropriate schedule of product admi...",HO
4,243849681,sepsis; febrile neutropenia,2024q4,DE,"{""patient"":""{""age"": ""61.0"", ""age_cod"": ""yr"", ""...","{""pt"": ""sepsis; febrile neutropenia"", ""uni_cod...",DE


In [17]:
oot["in_pt"] = [all(term in pts for term in pt.split("; ")) for pt in oot.pt]

In [20]:
sum(oot["in_pt"])/len(oot)

0.9959984195438535

# 7-level Ini_Code description
* Seven Level of - 

Code Levels:
* DE : Death
* LT : Life-Threatening
* HO : Hospitalization - Initial or Prolonged
* DS : Disability
* CA : Congenital Anomaly
* RI : Required Intervention to Prevent Permanent Impairment/Damage
* OT Other Serious (Important Medical Event)

# based on the serverity order

DE > LT > HO > DS > CA > RI > OT